In [1]:
import sys 
from SF_functions import *
from Header_Binnings import *
from params import *
import numpy as np 
import glob
from astropy.table import Table

In [2]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
import scipy
from scipy import stats
import scipy.optimize
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import time
import statistics 
from extinction import ccm89, apply
#import extinction
from astropy import table
from astropy.io import ascii 
from scipy.optimize import least_squares
import scipy.signal as mf 
from matplotlib.pyplot import show, plot
import sys 
import itertools
from error_routines import *
from numba import *
#from numba.typed import Dict 
from numba import types
import get_metadata
import json
import pandas as pd
from params import * 

In [3]:
import pickle

In [4]:
a_file = open("short_path_dict.pkl", "wb")
pickle.dump(get_metadata.short_path_dict, a_file)
a_file.close()

In [5]:
objects_path = '/mnt/c/Users/20xha/Documents/Caltech/Research/superfit/results/'
objects_list = glob.glob(objects_path+'*.csv') 

In [6]:
len(objects_list)

8835

In [7]:
sample = Table.read("/mnt/c/Users/20xha/Documents/Caltech/Research/superfit/ML_sample.ascii", format = "ascii")
sample.rename_column('col1', 'ZTF_Name')
sample.rename_column('col2', "Class")
sample.rename_column('col3', "redshift")
sample.rename_column('col8', "Version")

In [8]:
len(sample)

8959

In [9]:
csv = objects_list[0]

In [10]:
pd.read_csv(csv)

,OBJECT,GALAXY,SN,CONST_SN,CONST_GAL,Z,A_v,Frac(SN),Frac(gal),CHI2,CHI2/dof,CHI2/dof2,ln(prob),Band,Phase
0,AT2018hrg_20181103_P60_v1_10,/mnt/c/Users/20xha/Documents/Caltech/Research/...,II/1999em/LRIS+1999-11-08 00:00:00.00,1.219578,0.186343,-0.000152,1.8,0.744577,0.255423,703.86835,1.361448,0.002633,1.361448,B,-2451482.0
1,AT2018hrg_20181103_P60_v1_10,/mnt/c/Users/20xha/Documents/Caltech/Research/...,TDE H+He/ASASSN-14li/DIS+2014-12-02 12:08:44.00,0.135782,0.631786,-0.000168,-2.0,0.630945,0.369055,967.20575,1.896482,0.003719,1.896482,B,-2456940.0
2,AT2018hrg_20181103_P60_v1_10,/mnt/c/Users/20xha/Documents/Caltech/Research/...,II/2013fs/KAST+2013-10-26 08:06:13.00,1.076976,0.006475,-0.000152,0.4,0.992134,0.007866,1078.38490,2.004433,0.003726,2.004433,B,-2456563.0
3,AT2018hrg_20181103_P60_v1_10,/mnt/c/Users/20xha/Documents/Caltech/Research/...,II/2013ej/HET-LRS+2013-08-01 00:00:00.00,0.655982,0.183899,-0.000152,0.2,0.838233,0.161767,937.73550,1.871728,0.003736,1.871728,B,-2456508.0
4,AT2018hrg_20181103_P60_v1_10,/mnt/c/Users/20xha/Documents/Caltech/Research/...,II/2007od/KAST+2007-11-10 08:00:57.00,0.685210,0.040393,-0.000152,-0.4,0.967479,0.032521,1180.51680,2.194269,0.004079,2.194269,B,-2454354.0
5,AT2018hrg_20181103_P60_v1_10,/mnt/c/Users/20xha/Documents/Caltech/Research/...,Ia-pec/2000cx/KAST+2000-08-01 00:00:00.00,0.168436,0.466090,-0.000168,-1.4,0.668940,0.331060,1253.44820,2.329829,0.004331,2.329829,B,-2451753.0
6,AT2018hrg_20181103_P60_v1_10,/mnt/c/Users/20xha/Documents/Caltech/Research/...,II/2007od/LRIS+2007-11-12 04:35:02.00,0.591618,0.198926,-0.000152,-1.0,0.853623,0.146377,1182.73720,2.270129,0.004357,2.270129,B,-2454409.0


In [11]:
data = []
for csv in objects_list:
    data.append(pd.read_csv(csv))

KeyboardInterrupt: 

In [ ]:
data_np_superfit = np.asarray(data)

In [ ]:
len(data_np_superfit)

In [ ]:
data_np_superfit[0].iloc[0]["GALAXY"]

In [ ]:
data_np_superfit[0]

In [ ]:
np.save("data_superfit", data_np_superfit)

In [12]:
data_np_superfit_loaded = np.load("data_superfit.npy", allow_pickle = True)

In [13]:
test = data_np_superfit_loaded[14]

In [14]:
superfit_class = test.iloc[0]["SN"].split("/")[0]

In [15]:
name = test.iloc[0]["OBJECT"][0:test.iloc[0]["OBJECT"].rfind("_")]

In [16]:
name

'ZTF17aaazdba_20190304_P200_v1'

In [17]:
test.iloc[0]

OBJECT                        ZTF17aaazdba_20190304_P200_v1_10
GALAXY       /mnt/c/Users/20xha/Documents/Caltech/Research/...
SN                SLSN-I/2017egm/ALFOSC+2017-05-30 21:50:24.00
CONST_SN                                              0.244492
CONST_GAL                                             0.426122
Z                                                       0.0209
A_v                                                       -1.8
Frac(SN)                                              0.742861
Frac(gal)                                             0.257139
CHI2                                                   3039.63
CHI2/dof                                               4.93446
CHI2/dof2                                           0.00801049
ln(prob)                                               4.93446
Band                                                         B
Phase                                              -2.4579e+06
Name: 0, dtype: object

In [18]:
Classes_Final = Table(
                names=("Version", "Resolution", "Superfit_c", "Superfit_z", "Match", "Gal", "Frac(SN)", "Frac(gal)", "Phase", "Av"
                ),
                meta={"name": "Spectrum Results after Zooniverse"},
                dtype=("U64", "U64", "U64", "float32", "U64", "U64", "float32", "float32", "float32", "float32"
                      )
                )
for i in data_np_superfit_loaded:
    name = i.iloc[0]["OBJECT"][0:i.iloc[0]["OBJECT"].rfind("_")] + ".ascii"
    superfit_class = i.iloc[0]["SN"].split("/")[0]
    resolution = i.iloc[0]["OBJECT"]
    redshift = i.iloc[0]["Z"]
    match = i.iloc[0]["SN"]
    gal = i.iloc[0]["GALAXY"]
    fracsn = i.iloc[0]["Frac(SN)"]
    fracgal = i.iloc[0]["Frac(gal)"]
    phase = i.iloc[0]["Phase"]
    Av = i.iloc[0]["A_v"]
    row = [name, resolution, superfit_class, redshift, match, gal, fracsn, fracgal, phase, Av]
    Classes_Final.add_row(row)

In [19]:
Classes_Final.to_pandas().to_csv("superfit_classes.csv", index = False)